In [1]:
import numpy as np
from pyscf import gto, scf, lo, mp, cc

# mol = gto.Mole()
# mol.verbose = 4
# mol.atom = '''
# O1   -1.485163346097   -0.114724564047    0.000000000000
# H1   -1.868415346097    0.762298435953    0.000000000000
# H2   -0.533833346097    0.040507435953    0.000000000000
# O2    1.416468653903    0.111264435953    0.000000000000
# H3    1.746241653903   -0.373945564047   -0.758561000000
# H4    1.746241653903   -0.373945564047    0.758561000000
# '''
# mol.basis = 'sto6g'
# mol.build()

xyz_path = "/home/sharmagroup/sharmagroup/project/l7_dataset/geometry/c3a/B.xyz"
with open(xyz_path) as f:
    lines = f.readlines()

second_line = lines[1]
# charge   = int(second_line.split("charge=")[1].split()[0])
# spin     = int(second_line.split("mult=")[1].split()[0]) - 1
# mol_name = second_line.split()[0]
atoms    = "".join(lines[2:])

mol = gto.M(
    atom=atoms,
    basis="sto6g",
    verbose=4,
    unit="angstrom",
    max_memory=10000,
)

mf = scf.RHF(mol).density_fit()
mf.kernel()

# frozen = 2
# mymp = mp.MP2(mf, frozen=frozen)
# mymp.kernel()
# efull_mp2 = mymp.e_corr
# print(f'MP2 Corr = {efull_mp2:.8f}')

# mycc = cc.CCSD(mf, frozen=frozen)
# mycc.kernel()
# efull_ccsd = mycc.e_corr
# print(f'CCSD Corr = {efull_ccsd:.8f}')

# efull_t = mycc.ccsd_t()
# efull_ccsd_t = efull_ccsd + efull_t
# print(f'CCSD(T) Corr = {efull_ccsd_t:.8f}')

System: uname_result(system='Linux', node='sharmagroup-rn', release='6.17.0-29-generic', version='#29~24.04.1-Ubuntu SMP PREEMPT_DYNAMIC Mon May 11 10:30:58 UTC 2', machine='x86_64')  Threads 16
Python 3.12.13 | packaged by Anaconda, Inc. | (main, Mar 19 2026, 20:20:58) [GCC 14.3.0]
numpy 2.4.4  scipy 1.17.1  h5py 3.16.0
Date: Fri May 29 17:14:35 2026
PySCF version 2.12.1
PySCF path  /home/sharmagroup/sharmagroup/pyscf
GIT ORIG_HEAD 3d1768f5e33b144b606c3d2c81c12ee54d794501
GIT HEAD (branch master) f0861da51f017364d8bbaa20b742a94f3733305f

[ENV] OLD_PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge:
[ENV] PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge:/home/sharmagroup/sharmagroup/pyscf-forge:
[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 72
[INPUT] num. electrons = 342
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry False subgroup None
[INPUT] Mole.unit = angstrom
[INPUT] Symbol           X                Y             

np.float64(-2050.8000249153065)

In [ ]:
# print(f'MP2 Corr = {efull_mp2:.8f}')
# print(f'CCSD Corr = {efull_ccsd:.8f}')
# print(f'CCSD(T) Corr = {efull_ccsd_t:.8f}')

MP2 Corr = -0.07208475
CCSD Corr = -0.09974974
CCSD(T) Corr = -0.09996469


In [2]:
from pyscf.lno.tools import autofrag_iao
from pyscf import lo
import numpy as np
from pyscf.data import elements
# IAO localization
orbocc = mf.mo_coeff[:,elements.chemcore(mol):np.count_nonzero(mf.mo_occ)]
iao_coeff = lo.iao.iao(mol, orbocc)
iao_coeff = lo.orth.vec_lowdin(iao_coeff, mf.get_ovlp())
moliao = lo.iao.reference_mol(mol)
frag_lolist = autofrag_iao(moliao)
atom_msg = moliao.elements
print(atom_msg)

['C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H', 'H']


In [9]:
import numpy as np
from jax import random
from pyscf.lno import lnoccsd
from pyscf.lno import ulnoccsd
from collections.abc import Iterable

# from afqmc import config
from afqmc.lno_afqmc import prep
from afqmc.lno_afqmc import mod_lnoccsd

from functools import partial
import time, gc, pickle

# def run_afqmc(mf,
#               lo_coeff = None, 
#               lo_coeff_file = 'lo_coeff.npz',
#               frag_lolist = None,
#               nfrozen = 0,
#               thresh = 1e-6,
#               qmc_options = {}, 
#               chol_cut = 1e-5, 
#               target_sto_error = 1e-3, 
#               run_frg_list = None, 
#               atom_group = None,
#               ):

# mf
lo_coeff = iao_coeff
lo_coeff_file = 'lo_coeff.npz'
frag_lolist = autofrag_iao(moliao)
nfrozen = elements.chemcore(mol)
thresh = 1e-4
qmc_options = {}
chol_cut = 1e-5 
target_sto_error = 1e-3
run_frag_list = [14]
atom_group = moliao.elements

if lo_coeff is None:
    try:
        lo_coeff = np.load(lo_coeff_file)["lo_coeff"]
    except:
        raise ValueError(
            f"lo_coeff was not provided and could not be loaded from '{lo_coeff_file}'")

spin_type = prep.kind(lo_coeff)

if frag_lolist is None:
    if spin_type == "unrestricted":
        raise ValueError("frag_lolist must be provided for unrestricted LNO-AFQMC.")
    print("Fragment list not found. Asign every LO to a fragment.")
    frag_lolist = [[i] for i in range(lo_coeff.shape[1])]

if spin_type == "unrestricted":
    mlno = ulnoccsd.ULNOCCSD(mf, lo_coeff, frag_lolist, frozen=nfrozen).set(verbose=3)
    mf = mlno._scf
else:
    mlno = lnoccsd.LNOCCSD(mf, lo_coeff, frag_lolist, frozen=nfrozen).set(verbose=3)
mlno.lno_thresh = [thresh*10, thresh]
lno_thresh = mlno.lno_thresh
lno_type = ['1h','1h']
eris = mlno.ao2mo()

nfrag = len(frag_lolist)
if run_frag_list is None:
    run_frag_list = range(nfrag)

frag_lolist = [frag_lolist[i] for i in run_frag_list]

lno_pct_occ = [None, None]
lno_norb = [[None,None]] * nfrag

# seeds = random.randint(random.PRNGKey(qmc_options["seed"]),
#                        shape=(nfrag,), 
#                        minval=0, 
#                        maxval=100*nfrag
#                        )

qmc_options["max_error"] = target_sto_error / np.sqrt(nfrag)
trial_base = qmc_options.get("trial", "")

las_center = [None]*nfrag
las_size = np.zeros(nfrag, dtype='int32')
lno_emp2 = np.zeros(nfrag, dtype='float64')
lno_ecc  = np.zeros(nfrag, dtype='float64')
lno_eqmc = np.zeros(nfrag, dtype='float64')
lno_eqmc_err  = np.zeros(nfrag, dtype='float64')
ccsd_time = np.zeros(nfrag, dtype='float64')
qmc_time = np.zeros(nfrag, dtype='float64')

# Loop over fragment
for ifrag, loidx in enumerate(frag_lolist):
    print("\n")
    width = 80
    msg = f" {spin_type} LNO-FRAGMENT {run_frag_list[ifrag]+1}/{nfrag} "
    print(msg.center(width, '='))
    if atom_group is not None:
        atom_msg = f"{atom_group[ifrag]}"
        print(f"Center Atom {atom_msg}")

    if spin_type == "unrestricted":
        orbloc = [lo_coeff[0][:,loidx[0]], lo_coeff[1][:,loidx[1]]]
        lno_param = [
            [
                {
                    'thresh': (
                        lno_thresh[i][s] if isinstance(lno_thresh[i], Iterable)
                        else lno_thresh[i]
                    ),
                    'pct_occ': (
                        lno_pct_occ[i][s] if isinstance(lno_pct_occ[i], Iterable)
                        else lno_pct_occ[i]
                    ),
                    'norb': (
                        lno_norb[ifrag][i][s] if isinstance(lno_norb[ifrag][i], Iterable)
                        else lno_norb[ifrag][i]
                    ),
                } for i in [0, 1]
            ] for s in range(2)
        ]
    else:
        orbloc = lo_coeff[:,loidx]
        lno_param = [{
            'thresh': lno_thresh[i],
            'pct_occ': lno_pct_occ[i],
            'norb': lno_norb[ifrag][i]
            } for i in [0,1]]

    # M = <orbloc|canactocc> (M^dagger M)u = eu
    # u|canactocc> => orbtial in/out the space spanned by |orbloc>
    # uocc_loc = <lno_actocc|orbloc>
    lno_coeff, lno_frozen, uocc_loc, _ = mlno.make_las(eris, orbloc, lno_type, lno_param)
    # lno_coeff still connected to canonical mo_coeff unitarily

    if spin_type == "unrestricted":
        if uocc_loc[0].size > 0 and uocc_loc[1].size == 0:
            lno_elec_type = 'alpha'
        elif uocc_loc[0].size == 0 and uocc_loc[1].size > 0:
            lno_elec_type = 'beta'
        else:
            lno_elec_type = 'mixed'
        print(f'LNO-Frgament Spin Type = {lno_elec_type}')
    #     ao_message_a, ao_max_a = prep.ao_comp(mf, orbloc[0])
    #     ao_message_b, ao_max_b = prep.ao_comp(mf, orbloc[1])
    #     ao_message = ao_message_a + "\n" + ao_message_b
    #     ao_max = ao_max_a + ao_max_b
    # else:
    #     ao_message, ao_max = prep.ao_comp(mf, orbloc)

    mo_occ = mlno.mo_occ

    if spin_type == "unrestricted":
        lno_frozen, maskact = ulnoccsd.get_maskact(lno_frozen, [mo_occ[0].size, mo_occ[1].size])
        occidxa = mo_occ[0] > 1e-10
        occidxb = mo_occ[1] > 1e-10
        moidxa, moidxb = maskact
        nactocc_a = int(np.sum(moidxa & occidxa))
        nactvir_a = int(np.sum(moidxa & ~occidxa))
        nactocc_b = int(np.sum(moidxb & occidxb))
        nactvir_b = int(np.sum(moidxb & ~occidxb))
        nactocc = nactocc_a + nactocc_b
        nactvir = nactvir_a + nactvir_b
        print(f'LAS alpha: {nactocc_a} occupied, {nactvir_a} virtual')
        print(f'LAS beta:  {nactocc_b} occupied, {nactvir_b} virtual')
    else:
        lno_frozen, maskact = lnoccsd.get_maskact(lno_frozen, mo_occ.size)
        nactocc, nactvir = prep.las_size(mf, lno_frozen)
        print(f'LAS occupied orbitals: {nactocc}')
        print(f'LAS virtual orbitals: {nactvir}')

    if spin_type == "unrestricted":
        mcc = ulnoccsd.UCCSD(mf, mo_coeff=lno_coeff, frozen=lno_frozen).set(verbose=1)
    else:
        mcc = lnoccsd.CCSD(mf, mo_coeff=lno_coeff, frozen=lno_frozen).set(verbose=1)
    mcc._s1e = mlno._s1e
    mcc._h1e = mlno._h1e
    mcc._vhf = mlno._vhf
    if mlno.kwargs_imp is not None:
        mcc = mcc.set(**mlno.kwargs_imp)
    time0 = time.perf_counter()
    (eorb_mp2, eorb_cc), t1, t2 =\
        mod_lnoccsd.lnoccsd_kernel(mcc, lno_coeff, uocc_loc, mo_occ, maskact)
    time1 = time.perf_counter()
    lnocc_time = time1 - time0

    print(f"CCSD time: {lnocc_time:.6f} s")
    print(f'LNO-MP2 Orbital Energy: {eorb_mp2:.8f}')
    print(f'LNO-CCSD Orbital Energy: {eorb_cc:.8f}')

    if atom_group:
        las_center[ifrag] = atom_msg
    # else:
    #     las_center[ifrag] = ao_max
    las_size[ifrag] = nactocc + nactvir
    lno_emp2[ifrag] = eorb_mp2
    lno_ecc[ifrag] = eorb_cc
    ccsd_time[ifrag] = lnocc_time



======================== restricted LNO-FRAGMENT 15/72 =========================
Center Atom C
LAS occupied orbitals: 16
LAS virtual orbitals: 36
CCSD time: 1.035382 s
LNO-MP2 Orbital Energy: -0.05380960
LNO-CCSD Orbital Energy: -0.06080149


In [10]:
lno_active = np.array([i for i in range(mol.nao) if i not in lno_frozen])
print(lno_frozen)
print(lno_active)

[  0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17
  18  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35
  36  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53
  54  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71
  72  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88  89
  90  91  92  93  94  95  96  97  98  99 100 101 102 103 104 105 106 107
 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125
 126 127 128 129 130 131 132 133 134 135 136 137 138 139 140 141 142 143
 144 145 146 147 148 149 150 151 152 153 154 207 208 209 210 211 212 213
 214 215 216 217 218 219 220 221 222 223 224 225 226 227 228 229 230 231
 232 233 234 235 236 237 238 239 240 241 242 243 244 245 246 247 248 249
 250 251 252 253 254 255 256 257 258 259 260 261 262 263 264 265 266 267
 268 269 270 271 272 273 274 275 276 277 278 279 280 281 282 283 284 285
 286 287]
[155 156 157 158 159 160 161 162 163 164 

In [5]:
lno_coeff.shape

(288, 288)

In [30]:
lno_coeff[:,lno_active].shape

(14, 9)

In [ ]:
from pyscf.tools import molden

# with open('c3_iao_center.molden', 'w') as f:
#     molden.header(mol, f)
#     molden.orbital_coeff(mol, f, orbloc)

In [ ]:
# with open('c3_iao_las.molden', 'w') as f:
#     molden.header(mol, f)
#     molden.orbital_coeff(mol, f, lno_coeff[:,lno_active])

In [11]:
from pyscf.tools import cubegen
# For RHF closed-shell, occupation = 2 each
dm_center = orbloc @ orbloc.T
_ = cubegen.density(mol, 'c3_center_density.cube', dm_center)

In [12]:
dm_las = lno_coeff[:,lno_active] @ lno_coeff[:,lno_active].T
_ = cubegen.density(mol, 'c3_las_density.cube', dm_las)